# UNSW-NB15 Prediction Results and Evaluation

This notebook runs the saved real-data binary and multiclass models on the official test CSV, compares predictions with the test labels, displays metrics and confusion matrices, and saves prediction and metric CSVs under `predictions/`.

Run the cells from top to bottom using the project's Python environment. The test CSV is used only for evaluation here; do not use these evaluation results as training data.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'predict.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'predict.py').exists():
    raise FileNotFoundError('Could not find the project root containing src/predict.py')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, f1_score)

from src import config as cfg
from src.predict import predict_frame

DATA_PATH = cfg.DEFAULT_TEST
BINARY_MODEL_PATH = cfg.MODELS_DIR / 'binary_real' / 'best_model.joblib'
MULTICLASS_MODEL_PATH = cfg.MODELS_DIR / 'multiclass_real' / 'best_model.joblib'
OUTPUT_DIR = ROOT / 'predictions'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for required_path in (DATA_PATH, BINARY_MODEL_PATH, MULTICLASS_MODEL_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f'Required project file not found: {required_path}')

print(f'Project: {ROOT}')
print(f'Test data: {DATA_PATH}')
print(f'Prediction outputs: {OUTPUT_DIR}')

In [ ]:
test_df = pd.read_csv(DATA_PATH)
binary_artifact = joblib.load(BINARY_MODEL_PATH)
multiclass_artifact = joblib.load(MULTICLASS_MODEL_PATH)

print(f'Test rows: {len(test_df):,}')
print(f'Binary model: {binary_artifact["model_name"]}; labels: {binary_artifact["classes"]}')
print(f'Multiclass model: {multiclass_artifact["model_name"]}; labels: {multiclass_artifact["classes"]}')
print(test_df.head().to_string(index=False))

In [ ]:
binary_out, _ = predict_frame(binary_artifact, test_df)
multiclass_out, _ = predict_frame(multiclass_artifact, test_df)

binary_path = OUTPUT_DIR / 'binary_predictions.csv'
multiclass_path = OUTPUT_DIR / 'multiclass_predictions.csv'
binary_out.to_csv(binary_path, index=False)
multiclass_out.to_csv(multiclass_path, index=False)

print('Prediction counts:')
print(pd.concat({
    'Binary': binary_out['predicted_class'].value_counts(),
    'Multiclass': multiclass_out['predicted_class'].value_counts(),
}, axis=1).fillna(0).astype(int).to_string())
print(f'Saved: {binary_path}')
print(f'Saved: {multiclass_path}')

In [ ]:
def evaluate_predictions(y_true, y_pred, class_names):
    normal_name = cfg.NORMAL_NAME
    true_attack = pd.Series(y_true).to_numpy() != normal_name
    predicted_attack = pd.Series(y_pred).to_numpy() != normal_name
    attack_count = int(true_attack.sum())
    normal_count = int((~true_attack).sum())
    detected = int((true_attack & predicted_attack).sum())
    false_alarms = int((~true_attack & predicted_attack).sum())
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=class_names, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, labels=class_names, average='weighted', zero_division=0),
        'attack_detection_rate': detected / attack_count if attack_count else 0.0,
        'false_alarm_rate': false_alarms / normal_count if normal_count else 0.0,
    }

binary_classes = list(binary_artifact['classes'])
multiclass_classes = list(multiclass_artifact['classes'])

binary_true = test_df['label'].map({0: 'Normal', 1: 'Attack'}).astype(str)
multiclass_true = test_df['attack_cat'].astype(str).str.strip()
binary_pred = binary_out['predicted_class']
multiclass_pred = multiclass_out['predicted_class']

if not set(binary_true.unique()).issubset(binary_classes):
    raise ValueError(f'Binary truth labels do not match model labels: {binary_classes}')
if not set(multiclass_true.unique()).issubset(multiclass_classes):
    raise ValueError(f'Multiclass truth labels do not match model labels: {multiclass_classes}')

metrics = pd.DataFrame({
    'Binary': evaluate_predictions(binary_true, binary_pred, binary_classes),
    'Multiclass': evaluate_predictions(multiclass_true, multiclass_pred, multiclass_classes),
}).T
print(metrics.to_string(float_format=lambda value: f'{value:.4f}'))

metrics_path = OUTPUT_DIR / 'prediction_metrics.csv'
metrics.to_csv(metrics_path)
print(f'Saved evaluation metrics to {metrics_path}')

In [ ]:
print('Binary classification report')
print(classification_report(binary_true, binary_pred, labels=binary_classes,
                            target_names=binary_classes, digits=4, zero_division=0))
print('Multiclass classification report')
print(classification_report(multiclass_true, multiclass_pred, labels=multiclass_classes,
                            target_names=multiclass_classes, digits=4, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
ConfusionMatrixDisplay.from_predictions(
    binary_true, binary_pred, labels=binary_classes, display_labels=binary_classes,
    normalize='true', values_format='.2f', cmap='Blues', ax=axes[0], colorbar=False,
)
axes[0].set_title('Binary model — normalized confusion matrix')
ConfusionMatrixDisplay.from_predictions(
    multiclass_true, multiclass_pred, labels=multiclass_classes,
    display_labels=multiclass_classes, normalize='true', values_format='.2f',
    cmap='Blues', ax=axes[1], colorbar=False,
)
axes[1].set_title('Multiclass model — normalized confusion matrix')
axes[1].tick_params(axis='x', labelrotation=45)
fig.tight_layout()
confusion_path = OUTPUT_DIR / 'confusion_matrices.png'
fig.savefig(confusion_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved confusion-matrix figure to {confusion_path}')

In [ ]:
comparison = metrics[['accuracy', 'macro_f1', 'weighted_f1']].plot.bar(
    figsize=(9, 5), ylim=(0, 1), rot=0, title='Real-data test performance',
    ylabel='Score', xlabel='Task', grid=True,
)
comparison.legend(loc='lower right')
comparison.figure.tight_layout()
comparison_path = OUTPUT_DIR / 'metric_comparison.png'
comparison.figure.savefig(comparison_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved metric comparison to {comparison_path}')

print(pd.concat([
    binary_out[['predicted_class', 'confidence']].add_prefix('binary_'),
    multiclass_out[['predicted_class', 'confidence']].add_prefix('multiclass_'),
], axis=1).head(20).to_string(index=False))